# 01. Universo y target

## Objetivo
Definir el problema antes de construir variables:
- Cargar la base cruda y corregir la calidad de client_id.
- Definir las ventanas temporales de entrenamiento, lead y respuesta.
- Aplicar los filtros del universo.
- Construir el target y analizar su balance.

## Ventanas temporales
- Entrenamiento (observación): agosto 2018 a enero 2019. De acá salen todas las variables.
- Lead: febrero 2019. Brecha entre la foto del cliente y la respuesta, que simula el tiempo entre el scoring y la campaña.
- Respuesta: marzo y abril 2019. TGT = 1 si el cliente adoptó el producto en alguno de esos dos meses.

In [1]:
import pandas as pd

## Carga de datos

In [2]:
df = pd.read_csv('../data/raw/data.csv', sep='|', low_memory=False)
print(df.shape)

(238616, 77)


In [3]:
df['Month'].value_counts().sort_index()

Month
2018-08-01    26502
2018-09-01    26501
2018-10-01    26501
2018-11-01    26501
2018-12-01    26501
2019-01-01    26503
2019-02-01    26512
2019-03-01    26547
2019-04-01    26547
Name: count, dtype: int64

La base es una foto mensual por cliente: 9 meses (agosto 2018 a abril 2019) con unos 26.500 clientes por mes.

## Calidad de client_id
client_id se lee como texto porque la última fila del archivo es basura: un pie de exportación SQL, (238615 rows affected).

In [4]:
df['client_id'].map(type).value_counts()

client_id
<class 'str'>    238616
Name: count, dtype: int64

In [5]:
df.tail(2)[['client_id', 'Target', 'Month']]

,client_id,Target,Month
238614,5921880,0.0,2019-03-01
238615,(238615 rows affected),NaN,NaN


In [6]:
df['client_id'] = pd.to_numeric(df['client_id'], errors='coerce')
print('Filas sin client_id:', df['client_id'].isna().sum())

df = df[df['client_id'].notna()].copy()
df['client_id'] = df['client_id'].astype(int)

print('Clientes distintos:', df['client_id'].nunique())
print(df.shape)

Filas sin client_id: 1


Clientes distintos: 26560
(238615, 77)


Se convierten los client_id guardados como texto a número y se elimina la fila basura. Quedan 238.615 filas y 26.560 clientes.

## Filtros del universo

### 1. Clientes con los 9 meses de historia
Se exige historia completa para que todas las variables agregadas se calculen sobre la misma cantidad de meses.

In [7]:
con_9_meses = df['client_id'].value_counts().reset_index()
con_9_meses.columns = ['client_id', 'meses']
con_9_meses['meses'].value_counts()

meses
9    26483
2       38
8        9
7        9
3        9
4        6
1        6
Name: count, dtype: int64

### 2. Sin paquete activo a enero 2019
El producto objetivo es el paquete: se excluye a quien ya lo tiene en la fecha de corte, porque no puede adoptarlo.

In [8]:
sin_paquete = df[(df['Month'] == '2019-01-01') & (df['Package_Active'] == 'No')]
print(sin_paquete.shape[0])

26026


### 3. Sin tarjeta cobranding a enero 2019
Las tarjetas cobranding se emiten en acuerdo con un tercero (un comercio o una marca) y tienen condiciones comerciales propias, por lo que esos clientes no forman parte de la oferta del paquete.

In [9]:
sin_cobranding = df[(df['Month'] == '2019-01-01') & (df['CreditCard_CoBranding'] == 'No')]
print(sin_cobranding.shape[0])

23660


### Universo final
Intersección de los tres filtros.

In [10]:
universo = con_9_meses[con_9_meses['meses'] == 9][['client_id']] \
    .merge(sin_paquete[['client_id']], how='inner', on='client_id') \
    .merge(sin_cobranding[['client_id']], how='inner', on='client_id') \
    .drop_duplicates()

print(universo.shape)   # (23191, 1)

(23191, 1)


| Filtro | Clientes |
|---|---|
| Con 9 meses de historia | 26.483 |
| Sin paquete activo en enero 2019 | 26.026 |
| Sin tarjeta cobranding en enero 2019 | 23.660 |
| Universo (los tres a la vez) | 23.191 |

## Construcción del target
Se toma la ventana de respuesta (marzo y abril 2019). Un cliente del universo es TGT = 1 si tuvo Target = 1 en al menos uno de esos dos meses; el resto queda en TGT = 0.

In [11]:
df_pred = df[df['Month'].isin(['2019-03-01', '2019-04-01'])]

pred_tgt = df_pred[df_pred['Target'] == 1][['client_id']].drop_duplicates()
pred_tgt['TGT'] = 1

universo = universo.merge(pred_tgt, how='left', on='client_id')
universo['TGT'] = universo['TGT'].fillna(0).astype(int)

universo['TGT'].value_counts()

TGT
0    16368
1     6823
Name: count, dtype: int64

In [12]:
universo['TGT'].value_counts(normalize=True).round(3)

TGT
0    0.706
1    0.294
Name: proportion, dtype: float64

Conclusión – target: hay 6.823 clientes con TGT = 1 sobre 23.191 (29,4%). Es un desbalance moderado, por lo que no se aplica undersampling; si hace falta, en el modelado se pueden usar pesos por clase.

## Guardado

In [13]:
df.to_parquet('../data/processed/01_data.parquet', index=False)
universo.to_parquet('../data/processed/01_universo_tgt.parquet', index=False)